<a href="https://colab.research.google.com/github/KlauDCampos/Estudy-ML/blob/main/DataSet_Red_Atrib_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import pandas as pd
from google.colab import files
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.decomposition import PCA
from xgboost import XGBClassifier


In [ ]:
uploaded = files.upload()
arquivo = next(iter(uploaded))
df = pd.read_excel(arquivo)
print(df['label'].value_counts())
print(df.shape)


Saving 10.2.resultado_valoresReais_Multiclasse_excel_LabelEncoder_MinMaxScaler_label_binario.xlsx to 10.2.resultado_valoresReais_Multiclasse_excel_LabelEncoder_MinMaxScaler_label_binario.xlsx
label
0    163
1     33
Name: count, dtype: int64
(196, 21)


In [ ]:
df

,sexo,estudante_profissional,cor_raca,faixa_etaria,portador_de_necessidade_especiais,estado_civil,filhos,meio_de_transporte,oriundo_escola,atualmente_reside,...,tipo_de_moradia,estado_civil_dos_pais,renda_mensal,participacao_economica,cota,formacao_nivel_medio,cursou_outro_nivel_superior,label,zona,codigo
0,1,0,0.75,0.75,0,0.00,1,0.666667,0.5,0.8,...,1.0,0.25,0.50,0.25,0,0.0,1,1,0.012987,ZK9JVL32
1,0,0,0.75,0.25,0,0.50,0,0.666667,0.5,0.6,...,1.0,0.00,0.50,0.50,0,0.0,1,0,0.688312,6LD2OWS3
2,1,0,1.00,0.25,0,0.50,0,0.666667,1.0,0.6,...,1.0,0.25,0.50,0.75,1,1.0,0,0,0.688312,27JRYTA1
3,1,0,0.75,0.50,0,0.50,0,1.000000,1.0,1.0,...,1.0,0.00,0.25,0.75,1,0.0,0,0,0.298701,XJSRPOSC
4,1,0,1.00,0.75,0,0.00,1,0.666667,1.0,0.8,...,0.0,1.00,0.75,0.25,0,1.0,1,0,0.896104,LVL5L8J2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191,1,0,0.75,0.25,0,0.50,1,1.000000,0.5,0.6,...,1.0,0.75,0.25,0.75,0,0.0,1,0,0.792208,6XR3M6WZ
192,1,0,0.75,0.75,0,0.75,1,0.666667,0.5,0.8,...,0.0,0.00,0.25,0.25,0,0.0,0,0,0.623377,2818WGS8
193,1,1,0.25,0.25,0,0.50,0,1.000000,1.0,0.6,...,1.0,0.00,0.50,1.00,1,1.0,0,0,0.896104,NYBPWVEK
194,1,1,0.75,0.50,0,0.50,0,1.000000,1.0,0.6,...,1.0,0.25,0.25,0.00,1,0.0,0,0,0.896104,EJ8HQW7A


In [ ]:
df

print(df['label'].value_counts())
X,y = df.drop(['label','codigo'], axis=1), df['label']

pca = PCA(n_components=12)


X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42, stratify=y)
smote = SMOTE(
    sampling_strategy='auto',
    random_state=42
)


X_train_res, y_train_res = smote.fit_resample(
    X_train,
    y_train
)

X_train_pca = pca.fit_transform(X_train_res)
X_test_pca = pca.transform(X_test)
y_train_res_xgb = y_train_res - 1
y_test_xgb = y_test - 1
print("Variancia explicada por cada componente principal:")
print(pca.explained_variance_ratio_.round(3))
print("Soma da variância explicada:")
print(pca.explained_variance_ratio_.sum().round(3))




label
0    163
1     33
Name: count, dtype: int64
Variancia explicada por cada componente principal:
[0.218 0.161 0.119 0.085 0.067 0.061 0.057 0.05  0.03  0.026 0.021 0.02 ]
Soma da variância explicada:
0.915


In [ ]:
pipe_tree = Pipeline([
    ('dtc', DecisionTreeClassifier(max_depth=3, random_state=42)),
    ])
pipe_rf = Pipeline([
    ('rfc', RandomForestClassifier( n_estimators=100,random_state=42, max_depth=4))
])
pipe_knn = Pipeline([
    ('knn', KNeighborsClassifier(n_neighbors=11))
])
pipe_xgb = Pipeline([
    ('xgb', XGBClassifier(random_state=42,  eval_metric='mlogloss'))
])

In [ ]:
pipe_tree.fit(X_train_pca,y_train_res)
pipe_rf.fit(X_train_pca,y_train_res)
pipe_knn.fit(X_train_pca,y_train_res)
pipe_xgb.fit(X_train_pca,y_train_res)

print("Decision Tree")
print(pipe_tree.predict(X_test_pca))
print("Random Forest")
print(pipe_rf.predict(X_test_pca))
print("KNN")
print(pipe_knn.predict(X_test_pca))
print("XGB")
print(pipe_xgb.predict(X_test_pca))

Decision Tree
[1 1 1 1 0 1 0 1 0 1 0 1 0 1 1 1 0 1 0 1 1 0 1 0 0 0 1 0 0 1 1 1 0 0 0 0 0
 1 0 0]
Random Forest
[1 1 1 0 0 0 0 1 0 1 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 1 0 0 0 0 0
 1 0 0]
KNN
[1 1 1 0 0 0 1 1 0 1 0 0 1 0 1 1 1 0 1 0 0 1 0 1 0 0 1 0 0 1 1 1 0 0 0 0 0
 1 0 0]
XGB
[1 1 1 0 0 0 0 1 0 1 0 0 0 0 1 1 0 0 0 1 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0
 0 0 0]


In [ ]:
print("Decision Tree")
print(classification_report(y_test,pipe_tree.predict(X_test_pca)))
print("Random Forest")
print(classification_report(y_test,pipe_rf.predict(X_test_pca)))
print("KNN")
print(classification_report(y_test,pipe_knn.predict(X_test_pca)))
print("XGB")
print(classification_report(y_test,pipe_xgb.predict(X_test_pca)))

Decision Tree
              precision    recall  f1-score   support

           0       0.90      0.55      0.68        33
           1       0.25      0.71      0.37         7

    accuracy                           0.57        40
   macro avg       0.57      0.63      0.52        40
weighted avg       0.79      0.57      0.63        40

Random Forest
              precision    recall  f1-score   support

           0       0.86      0.76      0.81        33
           1       0.27      0.43      0.33         7

    accuracy                           0.70        40
   macro avg       0.57      0.59      0.57        40
weighted avg       0.76      0.70      0.72        40

KNN
              precision    recall  f1-score   support

           0       0.82      0.55      0.65        33
           1       0.17      0.43      0.24         7

    accuracy                           0.53        40
   macro avg       0.49      0.49      0.45        40
weighted avg       0.70      0.53      0.5

In [ ]:
y_pred_treino_tree = pipe_tree.predict(X_train_pca)
print("Decision Tree")
print("Acurácia no Treino:", accuracy_score(y_train_res, y_pred_treino_tree))
print("Acurácia no Teste: ", accuracy_score(y_test, pipe_tree.predict(X_test_pca)))
y_pred_treino_rf = pipe_rf.predict(X_train_pca)
print("Random Forest")
print("Acurácia no Treino:", accuracy_score(y_train_res, y_pred_treino_rf))
print("Acurácia no Teste: ", accuracy_score(y_test, pipe_rf.predict(X_test_pca)))
y_pred_treino_knn = pipe_knn.predict(X_train_pca)
print("KNN")
print("Acurácia no Treino:", accuracy_score(y_train_res, y_pred_treino_knn))
print("Acurácia no Teste: ", accuracy_score(y_test, pipe_knn.predict(X_test_pca)))
y_pred_treino_xgb = pipe_xgb.predict(X_train_pca)
print("XGB")
print("Acurácia no Treino:", accuracy_score(y_train_res, y_pred_treino_xgb))
print("Acurácia no Teste: ", accuracy_score(y_test, pipe_xgb.predict(X_test_pca)))

Decision Tree
Acurácia no Treino: 0.85
Acurácia no Teste:  0.575
Random Forest
Acurácia no Treino: 0.95
Acurácia no Teste:  0.7
KNN
Acurácia no Treino: 0.823076923076923
Acurácia no Teste:  0.525
XGB
Acurácia no Treino: 1.0
Acurácia no Teste:  0.725
